To run this script you need the optional project dependencies: `graphics`


In [ ]:
import polars as pl
import pandas as pd
from pathlib import Path
import numpy as np
import re

# plotting
import plotly.graph_objects as go
import plotly.express as px

# category cleaning
from sentence_transformers import SentenceTransformer
import hdbscan

# geo analysis
import geopandas as gpd
import json
from shapely.geometry import Point

# Load data

In [2]:
source = Path("../../","data","511_NY_Events__Beginning_2010_20260204.csv")
source

PosixPath('../../data/511_NY_Events__Beginning_2010_20260204.csv')

In [3]:
nyc_df = pl.scan_csv(source=source)
nyc_df.collect_schema()

Schema([('Event Type', String),
        ('Organization Name', String),
        ('Facility Name', String),
        ('Direction', String),
        ('City', String),
        ('County', String),
        ('State', String),
        ('Create Time', String),
        ('Close Time', String),
        ('Event Description', String),
        ('Responding Organization Id', String),
        ('Latitude', Float64),
        ('Longitude', Float64),
        ('Georeference', String)])

In [ ]:
all_cols = (
    nyc_df.select(
        ["Event Type", "Organization Name", "Facility Name", "Direction", "City", "County", "State",
        "Create Time", "Close Time", "Event Description", "Responding Organization Id",
        "Latitude", "Longitude", "Georeference"]
    )
)
all_cols.null_count().collect()

No missing data in `Latitude`, `Longitude`, `Georeference` ni `Create Time`. \
It only makes sense to consider rows where `Close Time` is present.
`County` is geographical, and can be inferred through `Georeference`.

For geo analysis, mainly coordinates were used. We only consider data in NYC. \
Further cleaning of the dataset using faulty `Responding Organization Id` is possible, we only do shallow analysis.

# Event description

## Responding Organization

In [ ]:
result = (
    nyc_df.select(
        ["Event Type", "Facility Name", "Direction", "City", "County", "State", "Georeference"]
    ).filter(pl.col("State").is_in(["NY","NJ","CT"]))
)

In [ ]:
state_counts = result.collect().with_columns(pl.len().over("State")).select(["State","len"]).unique().sort("len")
fig = go.Figure()
fig.add_trace(go.Bar(
    x=state_counts["len"],
    y=state_counts["State"],
    orientation="h"
))
fig.show()

In [ ]:
result_orgs = (
    nyc_df.select(
        ["Responding Organization Id"]
    )
)
org_counts = result_orgs.collect().with_columns(pl.len().over("Responding Organization Id")).select(["Responding Organization Id","len"]).unique().sort("len").tail(15)
fig = go.Figure()
fig.add_trace(go.Bar(
    x=org_counts["len"],
    y=org_counts["Responding Organization Id"],
    orientation="h"
))
fig.show()

In [ ]:
result_names = (
    nyc_df.select(
        ["Organization Name"]
    )
)
names_counts = result_names.collect().with_columns(pl.len().over("Organization Name")).select(["Organization Name","len"]).unique().sort("len").tail(15)
fig = go.Figure()
fig.add_trace(go.Bar(
    x=names_counts["len"],
    y=names_counts["Organization Name"],
    orientation="h"
))
fig.show()

In [ ]:
result = (
    nyc_df.select(
        ["Event Type", "Organization Name", "Facility Name", "Direction", "State",
        "Event Description", "Responding Organization Id"]
    ).filter(
        pl.col("State").is_null()
    )
)
org_counts = result.collect().with_columns(pl.len().over("Responding Organization Id")).select(["Responding Organization Id","len"]).unique().sort("len")
fig = go.Figure()
fig.add_trace(go.Bar(
    x=org_counts["len"],
    y=org_counts["Responding Organization Id"],
    orientation="h"
))
fig.show()

## Event type

In [5]:
nyc_df = nyc_df.with_columns(
    pl.col("Event Type")
    .str.to_lowercase()
    .str.strip_chars(" ")
    .alias("event_norm")
)
nyc_df = nyc_df.with_columns(
    pl.col("event_norm")
    .str.replace_all(r"[^\w\s]", " ")
    .str.replace_all(r"\s+", " ")
    .alias("event_norm")
)

In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

unique_events = (
    nyc_df.select("event_norm")
    .filter(pl.col("event_norm").is_not_null())
    .unique()
    .collect()
)
event_list = unique_events["event_norm"].to_list()

embeddings = model.encode(event_list, normalize_embeddings=True)
clusterer = hdbscan.HDBSCAN(min_cluster_size=12, min_samples=3, metric='euclidean')
labels = clusterer.fit_predict(embeddings)

event_clusters = pl.DataFrame({
    "event_norm": event_list,
    "Cluster": labels
})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 317.50it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
nyc_df = nyc_df.join(event_clusters.lazy(), on="event_norm", how="left")

In [8]:
# Count frequency of each event per cluster
cluster_names_df = (
    nyc_df
    .group_by(["Cluster", "Event Type"])
    .agg(pl.len().alias("freq"))
    .sort(["Cluster", "freq"], descending=[False, True])
    .group_by("Cluster")
    .agg(pl.col("Event Type").first().alias("Cluster Name"))  # most frequent event as cluster label
)

nyc_df = nyc_df.join(cluster_names_df, on="Cluster", how="left")
nyc_df = nyc_df.rename({"Cluster Name": "Event Bucket"})

In [9]:
desc_counts = (
    nyc_df.select(["Event Bucket"])
    .with_columns(pl.len().over("Event Bucket").alias("count"))
    .unique()
    .sort("count", descending=True)
    .collect()
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=desc_counts["count"],
    y=desc_counts["Event Bucket"],
    orientation="h"
))
fig.show()

In [10]:
buckets_map = {
    "traffic_accident": [
        "accident", "vehicle", "trailer", "diesel", 
        "overturn", "jack-knifed", "incident", "fire department",
        "police", "truck", "crash", "collision"
    ],
    "construction": [
        "construction", "repair", "break", "paving", "sealing", 
        "pole", "power line", "tree", "bridge", "sinkhole",
        "drainage", "pothole", "maintenance", "roadwork",
        "manhole", "debris", "structure", "sewer", "building"
    ],
    "weather": [
        "rain", "snow", "weather", "wet", "wind", 
        "sun glare", "storm"
    ],
    "cultural": [
        "event", "game", "match", "sport",
        "concert", "holiday", "basketball", "baseball", 
        "hockey", "soccer", "festival"
    ],
}

event_expr = None

for category, keywords in buckets_map.items():
    if not keywords:
        continue
        
    pattern = "|".join(re.escape(k) for k in keywords)
    condition = pl.col("Event Bucket").str.contains(pattern)

    if event_expr is None:
        event_expr = pl.when(condition).then(pl.lit(category))
    else:
        event_expr = event_expr.when(condition).then(pl.lit(category))

event_expr = event_expr.otherwise(pl.lit("other")).alias("Event Category")

nyc_df = nyc_df.with_columns(event_expr)

In [11]:
categ_counts = (
    nyc_df.select(["Event Category"])
    .with_columns(pl.len().over("Event Category").alias("count"))
    .unique()
    .sort("count", descending=True)
    .collect()
)

color_map = {
    "construction": "#2ca02c",
    "traffic_accident": "#b22222",
    "cultural": "#9932cc",
    "weather": "#4169e1",
    "other": "#778899"
}

colors = [
    color_map.get(cat)
    for cat in categ_counts["Event Category"]
]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=categ_counts["count"],
    y=categ_counts["Event Category"],
    orientation="h",
    marker=dict(color=colors)
))
fig.update_layout(
    title="Total Events by type",
    margin=dict(l=4, r=8, t=32, b=8),
)
fig.show()

In [12]:
validation_df = (
    nyc_df
    .group_by("Event Category")
    .agg(
        pl.col("Event Bucket").unique().sort()
    )
    .sort("Event Category")
)

validation_dict = {
    row["Event Category"]: row["Event Bucket"]
    for row in validation_df.collect().to_dicts()
}
validation_dict

{'construction': ['bridge construction',
  'bridge demolition',
  'bridge painting',
  'bridge work',
  'construction',
  'crack sealing',
  'culvert repairs',
  'debris spill',
  'downed pole',
  'downed power lines',
  'drainage improvements',
  'drainage improvements, guard rail repairs',
  'drainage improvements, sign work, traffic signal repairs, utility work',
  'expansion joint repairs',
  'fallen trees,downed power lines',
  'guard rail repairs',
  'milling, paving operations',
  'milling, pothole repair',
  'milling, repaving',
  'overhead sign repair',
  'overnight roadwork, paving operations',
  'paving operations',
  'pothole repairs',
  'road construction,delays',
  'road maintenance operations',
  'road maintenance operations,narrow lanes',
  'sinkhole',
  'traffic signal repairs',
  'tree trimming',
  'watermain break'],
 'cultural': ['concert,traffic much heavier than normal,delays',
  'special event'],
 'other': [None,
  'delays, pockets of volume',
  'installation of 

# Temporal analysis

In [61]:
nyc_times = (
    nyc_df
    .select(["Create Time", "Close Time", "Event Category"])
    .filter(
        pl.col("Create Time").is_not_null() &
        pl.col("Close Time").is_not_null()
    ).with_columns(
        pl.col("Close Time").str.strptime(pl.Datetime, format="%m/%d/%Y %I:%M:%S %p").alias("Close Time_DT"),
        pl.col("Create Time").str.strptime(pl.Datetime, format="%m/%d/%Y %I:%M:%S %p").alias("Create Time_DT"),
    ).with_columns(
        ((pl.col("Close Time_DT") - pl.col("Create Time_DT")).cast(pl.Int64) / 1_000_000 / 60).alias("time_diff") # mins
    ).filter(
        pl.col("time_diff").is_not_null() &
        (pl.col("time_diff") > 0)
    ).select(["time_diff", "Event Category"])
)

construction_data = (
    nyc_times
    .with_columns(
        pl.col("time_diff") / 60 # hours
    )
    .filter(
        (pl.col("Event Category") == "construction") &
        (pl.col("time_diff") < 24 * 7)
    )
    .collect()
    .to_pandas()
)["time_diff"]

category_data = (
    nyc_times.filter(
        (pl.col("Event Category") != "construction") &
        (pl.col("time_diff") < 24 * 7)
    )
    .collect()
    .to_pandas()
)

In [68]:
x_min = 0
x_max = 24*3
x_min_show = 0
x_max_show = 32

construction_data = construction_data[
    (construction_data >= x_min) &
    (construction_data <= x_max)
]

nbins = 24
counts, bin_edges = np.histogram(construction_data, bins=nbins)
probabilities = counts / counts.sum()
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=bin_centers,
    y=probabilities,
    mode="lines",
    name="Construction",
    line_color="#2ca02c"
))

fig.update_layout(
    title="Duration Distribution of Construction Events",
    xaxis_title="Duration (Hours)",
    yaxis_title="Normalized Counts",
    xaxis=dict(range=[x_min_show, x_max_show]),
    template="plotly",
    showlegend=False
)

fig.show()

In [71]:
categories = category_data['Event Category'].unique()
colors = {
    "traffic_accident": "#b22222",
    "cultural": "#9932cc",
    "weather": "#4169e1",
    "other": "#778899"
}

fig = go.Figure()

nbins = 8
all_values = category_data["time_diff"]
bin_edges = np.linspace(all_values.min(), all_values.max(), nbins + 1)

for categ in categories:
    cat_df = category_data[
        category_data['Event Category'] == categ
    ]["time_diff"]

    counts, _ = np.histogram(cat_df, bins=bin_edges)
    probs = counts / counts.sum()
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    fig.add_trace(
        go.Scatter(
            x=bin_centers,
            y=probs,
            mode="lines",
            name=categ,
            line=dict(color=colors.get(categ), width=3)
        )
    )

fig.update_layout(
    title="Duration Distribution of Events by Category",
    xaxis_title="Duration (Minutes)",
    yaxis_title="Normalized Counts",
    template="plotly",
)

fig.show()


# Geo analysis

In [ ]:
points_filtered = (
    nyc_df
    .select(["State", "City", "Latitude", "Longitude"])
    .filter(
        pl.col("Latitude").is_not_null() &
        pl.col("Longitude").is_not_null() &
        (pl.col("State") == "NY")
    )
    .collect()
    .to_pandas()
)

points_geo = gpd.GeoDataFrame(
    points_filtered,
    geometry=gpd.points_from_xy(points_filtered["Longitude"], points_filtered["Latitude"]),
    crs="EPSG:4326"
)

In [ ]:
geojson_path = Path("../..", "data", "NYC_Taxi_Zones_20260207.geojson")

with open(geojson_path) as f:
    geojson = json.load(f)

polygons_geo = gpd.GeoDataFrame.from_features(geojson["features"])
polygons_geo = polygons_geo.set_crs("EPSG:4326")

In [ ]:
def plot_events_by_borough(points_geo, polygons, title : str, map_color : str):
    joined = gpd.sjoin(
        points_geo,
        polygons,
        how="left",
        predicate="within"
    )

    density = (
        joined
        .groupby("zone")
        .size()
        .reset_index(name="count")
    )

    polygons_count = polygons.merge(density, on="zone", how="left")

    fig = px.choropleth_map(
        polygons_count,
        geojson=geojson,
        locations="zone",
        featureidkey="properties.zone",
        color="count",
        color_continuous_scale=map_color,
        hover_name="zone",
        map_style="carto-positron",
        zoom=9,
        center={"lat": 40.7128, "lon": -74.0060},
    )

    fig.update_layout(title=title)
    fig.update_layout(
        margin=dict(l=8, r=0, t=32, b=8),
    )
    fig.show()

In [ ]:
plot_events_by_borough(
    points_geo, polygons_geo, 
    "Total Events by Borough", "algae",
    )

In [ ]:
filtered_construction = (
    nyc_df
    .select(["State", "City", "Latitude", "Longitude", "Event Category"])
    .filter(
        pl.col("Latitude").is_not_null() &
        pl.col("Longitude").is_not_null() &
        (pl.col("State") == "NY") &
        (pl.col("Event Category") == "construction")
    )
    .collect()
    .to_pandas()
)

construction_geo = gpd.GeoDataFrame(
    filtered_construction,
    geometry=gpd.points_from_xy(filtered_construction["Longitude"], filtered_construction["Latitude"]),
    crs="EPSG:4326"
)

plot_events_by_borough(
    construction_geo, polygons_geo, 
    "Construction Events by Borough", "algae",
    )

In [ ]:
filtered_accident = (
    nyc_df
    .select(["State", "City", "Latitude", "Longitude", "Event Category"])
    .filter(
        pl.col("Latitude").is_not_null() &
        pl.col("Longitude").is_not_null() &
        (pl.col("State") == "NY") &
        (pl.col("Event Category") == "traffic_accident")
    )
    .collect()
    .to_pandas()
)

accident_geo = gpd.GeoDataFrame(
    filtered_accident,
    geometry=gpd.points_from_xy(filtered_accident["Longitude"], filtered_accident["Latitude"]),
    crs="EPSG:4326"
)

plot_events_by_borough(
    accident_geo, polygons_geo, 
    "Traffic Accidents by Borough", "reds",
    )

In [ ]:
filtered_cultural = (
    nyc_df
    .select(["State", "City", "Latitude", "Longitude", "Event Category"])
    .filter(
        pl.col("Latitude").is_not_null() &
        pl.col("Longitude").is_not_null() &
        (pl.col("State") == "NY") &
        (pl.col("Event Category") == "cultural")
    )
    .collect()
    .to_pandas()
)

cultural_geo = gpd.GeoDataFrame(
    filtered_cultural,
    geometry=gpd.points_from_xy(filtered_cultural["Longitude"], filtered_cultural["Latitude"]),
    crs="EPSG:4326"
)

plot_events_by_borough(
    cultural_geo, polygons_geo, 
    "Cultural Events by Borough", "purp",
    )

In [ ]:
filtered_weather = (
    nyc_df
    .select(["State", "City", "Latitude", "Longitude", "Event Category"])
    .filter(
        pl.col("Latitude").is_not_null() &
        pl.col("Longitude").is_not_null() &
        (pl.col("State") == "NY") &
        (pl.col("Event Category") == "weather")
    )
    .collect()
    .to_pandas()
)

weather_geo = gpd.GeoDataFrame(
    filtered_weather,
    geometry=gpd.points_from_xy(filtered_weather["Longitude"], filtered_weather["Latitude"]),
    crs="EPSG:4326"
)

plot_events_by_borough(
    weather_geo, polygons_geo, 
    "Weather Events by Borough", "teal",
    )